# 信号处理顺序诊断 Notebook

这个 notebook 演示如何在 Jupyter 中比较多个 CSI 预处理顺序，例如“先降噪再校相”和“先校相再降噪”。核心判断逻辑在 `wsdp.diagnostics` 中；notebook 只负责准备数据、定义候选、调用 API、展示结果。

In [ ]:
# Module 1: 环境和导入
from pathlib import Path
import json
import sys

# 让 notebook 可从仓库根目录或 examples/ 目录运行。
for candidate in (Path.cwd() / "src", Path.cwd().parent / "src"):
    if candidate.exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

import numpy as np
import pandas as pd
from IPython.display import Image, display

from wsdp.diagnostics import compare_pipeline_candidates

output_dir = Path("output/pipeline_order_selection_demo")
print(f"Diagnostics will be written to: {output_dir.resolve()}")

In [ ]:
# Module 2: 生成一个带随机相位扰动的复数 CSI 样本
# 真实使用时，把 raw_csi 和候选结果替换为你的 .npy / reader 输出即可。
def synthetic_phase_jitter_csi(T=128, F=16, A=1):
    t = np.arange(T)
    subcarriers = np.linspace(-1.0, 1.0, F)
    base = np.ones((T, F, A), dtype=complex)
    motion = 0.2 * np.exp(1j * 2 * np.pi * 0.08 * t)[:, None, None]
    clean = base + motion * np.linspace(0.8, 1.2, F)[None, :, None]

    common = 1.2 * np.sin(2 * np.pi * 0.11 * t)
    slope = 0.9 * np.sin(2 * np.pi * 0.07 * t + 0.3)
    phase_error = common[:, None, None] + slope[:, None, None] * subcarriers[None, :, None]
    raw = clean * np.exp(1j * phase_error)
    return raw, clean

raw_csi, phase_stabilized_csi = synthetic_phase_jitter_csi()
print("raw_csi shape:", raw_csi.shape)

In [ ]:
# Module 3: 定义候选流程结果
# 这里用生成数据构造两个候选结果。真实场景中，替换为 denoise()/calibrate()/自定义 pipeline 的输出。
candidates = {
    "denoise_then_calibrate": raw_csi,
    "calibrate_then_denoise": phase_stabilized_csi,
}

In [ ]:
# Module 4: 调用解耦后的核心 API
result = compare_pipeline_candidates(
    raw_csi,
    candidates,
    output_dir=output_dir,
    sampling_rate=1.0,
    motion_band=(0.04, 0.15),
    n_fft=32,
    hop_length=16,
)

print(json.dumps({k: v for k, v in result.items() if k != "comparison_rows"}, indent=2))

In [ ]:
# Module 5: 查看候选比较表
comparison = pd.read_csv(result["comparison_csv"])
comparison.sort_values("quality_score", ascending=False)

In [ ]:
# Module 6: 展示诊断图片
for name, path in result["diagnostic_paths"].items():
    print(name, path)
    display(Image(filename=path))

## 如何替换为真实数据

1. 将 `raw_csi` 替换为 reader 或 `.npy` 加载得到的原始复数 CSI。
2. 用你的算法生成多个候选结果，例如先做轻量异常点清洗、先相位校准、先带通滤波等。
3. 保持 `candidates` 的每个数组形状与 `raw_csi` 一致。
4. 调整 `sampling_rate` 和 `motion_band`，使其匹配你的采样率和目标运动频段。
5. 如果 `confidence` 是 `inconclusive`，不要强行选流程；优先查看图片并增加静止片段或下游任务验证。